# 17 — Completeness audit, CheXbert labels, and RadGraph F1

Run in a fresh GPU kernel after generation to free model memory. This uses the existing
CheXbert installation/checkpoint and frozen mapping, with negation/uncertainty sanity checks.
Scoring is chunked and resumable. Missing RadGraph scores are errors, never zeros.
No new human-validation claim is made, and the old validation gate is not modified.

In [ ]:
from pathlib import Path
import os, sys, json

candidate = Path(os.environ.get("JAMIA_RERUN_DIR", Path.cwd())).resolve()
if not (candidate / "rerun_config.json").is_file() and (candidate.parent / "rerun_config.json").is_file():
    candidate = candidate.parent
RERUN_DIR = candidate
if not (RERUN_DIR / "rerun_config.json").is_file():
    raise FileNotFoundError("Copy rerun_config.example.json to rerun_config.json, configure paths, and set JAMIA_RERUN_DIR to the repository root")
implementation = (RERUN_DIR / "src" / "rerun_code").resolve()
sys.path[:] = [entry for entry in sys.path if Path(entry or ".").resolve() != implementation]
sys.path.insert(0, str(RERUN_DIR / "src"))
from rerun_code.config import load_config
_, CONFIG = load_config(RERUN_DIR)
RESULTS_ROOT = Path(os.environ.get("JAMIA_OUTPUT_ROOT", CONFIG["output_root"]))
CONFIG["output_root"] = str(RESULTS_ROOT)
RUN_NAME = os.environ.get("JAMIA_ABLATION_RUN", "matched_input_v1")
ABLATION_ROOT = RESULTS_ROOT / "post_rerun" / "modality_ablation" / RUN_NAME
print("Code:", RERUN_DIR)
print("Existing inputs:", RESULTS_ROOT)
print("NEW output:", ABLATION_ROOT)


In [ ]:
from rerun_code.modality_ablation import complete_frame, score_reports
output, PLAN, generation = complete_frame(RERUN_DIR, CONFIG, RUN_NAME)
print("Complete paired generation records:", len(generation))
# Keep this fixed when resuming the same run.
CHUNK_SIZE = 128
scored = score_reports(RERUN_DIR, CONFIG, RUN_NAME, chunk_size=CHUNK_SIZE)
print("Scored records:", len(scored))
print("Saved:", output / "scored_records.json")